<h1>Model Ensemdling: Hill Climbing</h1>

In [55]:
import pandas as pd; pd.set_option('display.max_columns', 100)
import numpy as np

import re

import matplotlib.pyplot as plt; plt.style.use('ggplot')
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder, FunctionTransformer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split, cross_val_score, StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, roc_curve, RocCurveDisplay, cohen_kappa_score, log_loss, f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from collections import Counter
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from sklearn.manifold import TSNE
import optuna

def gini_coefficient(y_true, y_pred):
    return 2 * roc_auc_score(y_true, y_pred) - 1

In [2]:
X_train = pd.read_csv('../data/train_X.csv').fillna(0) 
test_cv = pd.read_csv('../data/test1_X.csv').fillna(0)
Y_train = pd.read_csv('../data/train_y.csv')['default6']

<h4>Дропаем колонки с датами и айдишниками (на них разделяем в кросс-валидации)</h4>

In [3]:
groups = X_train['contract_id']
X_train = X_train.drop(['report_date', 'contract_date', 'contract_id'], axis=1)
test_cv = test_cv.drop(['report_date', 'contract_date', 'contract_id'], axis=1)

<h4>Ищем параметры для моделей с Optuna</h4>

In [ ]:
n_splits = 5


def optimize_rf(trial):
    max_depth = trial.suggest_int('max_depth', 5, 15)
    min_samples_split = trial.suggest_int('min_samples_split', 5, 30)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 15)

    model = RandomForestClassifier(
        n_estimators=500,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )

    skf = StratifiedKFold(n_splits=n_splits)
    score = cross_val_score(model, X_train, Y_train, cv=skf, n_jobs=-1, groups=groups, scoring='roc_auc').mean()
    return score

def optimize_et(trial):
    max_depth = trial.suggest_int('max_depth', 5, 15)
    min_samples_split = trial.suggest_int('min_samples_split', 5, 30)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 15)

    model = ExtraTreesClassifier(
        n_estimators=500,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )

    skf = StratifiedKFold(n_splits=n_splits)
    score = cross_val_score(model, X_train, Y_train, cv=skf, n_jobs=-1, groups=groups, scoring='roc_auc').mean()
    return score

def optimize_hist(trial):
    l2_regularization = trial.suggest_loguniform('l2_regularization', 1e-3, 1e-1)
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-3, 1e-1)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    max_bins = trial.suggest_int('max_bins', 100, 255)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 5, 30)
    max_leaf_nodes = trial.suggest_int('max_leaf_nodes', 10, 30)

    model = HistGradientBoostingClassifier(
        l2_regularization=l2_regularization,
        learning_rate=learning_rate,
        max_iter=500,
        max_depth=max_depth,
        max_bins=max_bins,
        min_samples_leaf=min_samples_leaf,
        max_leaf_nodes=max_leaf_nodes,
        random_state=42
    )

    skf = StratifiedKFold(n_splits=n_splits)
    score = cross_val_score(model, X_train, Y_train, cv=skf, n_jobs=-1, groups=groups, scoring='roc_auc').mean()
    return score

# Repeat for LGBM, XGBoost, and CatBoost, adapting their specific hyperparameters
def optimize_lgbm(trial):
    max_depth = trial.suggest_int('max_depth', 3, 15)
    num_leaves = trial.suggest_int('num_leaves', 20, 150)
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-4, 1e-1)
    reg_alpha = trial.suggest_loguniform('reg_alpha', 1e-2, 10.0)
    reg_lambda = trial.suggest_loguniform('reg_lambda', 1e-2, 10.0)
    subsample = trial.suggest_uniform('subsample', 0.6, 1.0)
    colsample_bytree = trial.suggest_uniform('colsample_bytree', 0.6, 1.0)

    model = LGBMClassifier(
        objective='binary',
        n_estimators=500,
        max_depth=max_depth,
        num_leaves=num_leaves,
        learning_rate=learning_rate,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        random_state=42
    )

    skf = StratifiedKFold(n_splits=n_splits)
    score = cross_val_score(model, X_train, Y_train, cv=skf, n_jobs=-1, groups=groups, scoring='roc_auc').mean()
    return score

def optimize_xgb(trial):
    max_depth = trial.suggest_int('max_depth', 3, 15)
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-4, 1e-1)
    gamma = trial.suggest_loguniform('gamma', 1e-3, 1e-1)
    min_child_weight = trial.suggest_int('min_child_weight', 1, 10)
    subsample = trial.suggest_uniform('subsample', 0.6, 1.0)
    colsample_bytree = trial.suggest_uniform('colsample_bytree', 0.6, 1.0)
    reg_alpha = trial.suggest_loguniform('reg_alpha', 1e-2, 10.0)
    reg_lambda = trial.suggest_loguniform('reg_lambda', 1e-2, 10.0)

    model = XGBClassifier(
        objective='binary:logistic',
        tree_method='hist',
        n_estimators=500,
        max_depth=max_depth,
        learning_rate=learning_rate,
        gamma=gamma,
        min_child_weight=min_child_weight,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    )

    skf = StratifiedKFold(n_splits=n_splits)
    score = cross_val_score(model, X_train, Y_train, cv=skf, n_jobs=-1, groups=groups, scoring='roc_auc').mean()
    return score

def optimize_cat(trial):
    depth = trial.suggest_int('depth', 4, 10)
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-4, 1e-1)
    l2_leaf_reg = trial.suggest_loguniform('l2_leaf_reg', 1e-3, 10)
    bagging_temperature = trial.suggest_uniform('bagging_temperature', 0, 1)
    border_count = trial.suggest_int('border_count', 1, 255)
    random_strength = trial.suggest_loguniform('random_strength', 1e-9, 10)

    model = CatBoostClassifier(
        loss_function='Logloss',
        iterations=500,
        depth=depth,
        learning_rate=learning_rate,
        l2_leaf_reg=l2_leaf_reg,
        bagging_temperature=bagging_temperature,
        border_count=border_count,
        random_strength=random_strength,
        verbose=False,
        random_seed=42,
        task_type='CPU'
    )

    skf = StratifiedKFold(n_splits=n_splits)
    score = cross_val_score(model, X_train, Y_train, cv=skf, n_jobs=-1, groups=groups, scoring='roc_auc').mean()
    return score

study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(optimize_rf, n_trials=50)
print("Best parameters for Random Forest: ", study_rf.best_params)
np.save('models_parameters/best_params_rf.npy', study_rf.best_params)

study_et = optuna.create_study(direction='maximize')
study_et.optimize(optimize_et, n_trials=50)
print("Best parameters for Extra Trees: ", study_et.best_params)
np.save('models_parameters/best_params_et.npy', study_et.best_params)

study_hist = optuna.create_study(direction='maximize')
study_hist.optimize(optimize_hist, n_trials=50)
print("Best parameters for HistGradientBoosting: ", study_hist.best_params)
np.save('models_parameters/best_params_hist.npy', study_hist.best_params)

study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(optimize_lgbm, n_trials=50)
print("Best parameters for LightGBM: ", study_lgbm.best_params)
np.save('models_parameters/best_params_lgbm.npy', study_lgbm.best_params)

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(optimize_xgb, n_trials=50)
print("Best parameters for XGBoost: ", study_xgb.best_params)
np.save('models_parameters/best_params_xgb.npy', study_xgb.best_params)

study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(optimize_cat, n_trials=50)
print("Best parameters for CatBoost: ", study_cat.best_params)
np.save('models_parameters/best_params_cat.npy', study_cat.best_params)

In [4]:
def hill_climbing(x, y, x_test):
    
    scores = {}
    for col in x.columns:
        scores[col] = roc_auc_score(y, x[col])

    scores = {k: v for k, v in sorted(scores.items(), key = lambda item: item[1], reverse = True)}

    x = x[list(scores.keys())]
    x_test = x_test[list(scores.keys())]

    STOP = False
    current_best_ensemble = x.iloc[:,0]
    current_best_test_preds = x_test.iloc[:,0]
    MODELS = x.iloc[:,1:]
    weight_range = np.arange(-0.5, 0.51, 0.01) 
    history = [roc_auc_score(y, current_best_ensemble)]
    j = 0

    while not STOP:
        j += 1
        potential_new_best_cv_score = roc_auc_score(y, current_best_ensemble)
        k_best, wgt_best = None, None
        for k in MODELS:
            for wgt in weight_range:
                potential_ensemble = (1 - wgt) * current_best_ensemble + wgt * MODELS[k]
                cv_score = roc_auc_score(y, potential_ensemble)
                if cv_score > potential_new_best_cv_score:
                    potential_new_best_cv_score = cv_score
                    k_best, wgt_best = k, wgt

        if k_best is not None:
            current_best_ensemble = (1 - wgt_best) * current_best_ensemble + wgt_best * MODELS[k_best]
            current_best_test_preds = (1 - wgt_best) * current_best_test_preds + wgt_best * x_test[k_best]
            MODELS.drop(k_best, axis = 1, inplace = True)
            if MODELS.shape[1] == 0:
                STOP = True
            history.append(potential_new_best_cv_score)
        else:
            STOP = True
        
    hill_ens_pred_1 = current_best_ensemble
    hill_ens_pred_2 = current_best_test_preds
    
    return [hill_ens_pred_1, hill_ens_pred_2]

In [5]:
best_params_rf = np.load('models_parameters/best_params_rf.npy', allow_pickle=True).item()
best_params_et = np.load('models_parameters/best_params_et.npy', allow_pickle=True).item()
best_params_hist = np.load('models_parameters/best_params_hist.npy', allow_pickle=True).item()
best_params_lgbm = np.load('models_parameters/best_params_lgbm.npy', allow_pickle=True).item()
best_params_xgb = np.load('models_parameters/best_params_xgb.npy', allow_pickle=True).item()
best_params_cat = np.load('models_parameters/best_params_cat.npy', allow_pickle=True).item()

In [6]:
X, Y = X_train.copy(), Y_train.copy()

In [ ]:
ens_cv_scores, ens_preds = list(), list()
hill_ens_cv_scores, hill_ens_preds = list(), list()

sk = StratifiedGroupKFold(n_splits=25, random_state=42, shuffle=True)

for i, (train_idx, test_idx) in enumerate(sk.split(X, Y, groups)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    Y_train, Y_test = Y.iloc[train_idx], Y.iloc[test_idx]
    print('----------------------------------------------------------')
    
    ## RF ##
    RF_md = RandomForestClassifier(
        n_estimators=500,
        max_depth=best_params_rf['max_depth'],
        min_samples_split=best_params_rf['min_samples_split'],
        min_samples_leaf=best_params_rf['min_samples_leaf'],
        random_state=42
    ).fit(X_train, Y_train)
    
    RF_pred = RF_md.predict_proba(X_test)[:, 1]
    RF_score = roc_auc_score(Y_test, RF_pred)
    print('Fold', i, '==> RF oof ROC-AUC score is ==>', RF_score)
    RF_pred_test = RF_md.predict_proba(test_cv)[:, 1]
    
    ## Extra Trees ##
    ET_md = ExtraTreesClassifier(
        n_estimators=500,
        max_depth=best_params_et['max_depth'],
        min_samples_split=best_params_et['min_samples_split'],
        min_samples_leaf=best_params_et['min_samples_leaf'],
        random_state=42
    ).fit(X_train, Y_train)

    ET_pred = ET_md.predict_proba(X_test)[:, 1]
    ET_score = roc_auc_score(Y_test, ET_pred)
    print('Fold', i, '==> ET oof ROC-AUC score is ==>', ET_score)
    ET_pred_test = ET_md.predict_proba(test_cv)[:, 1]

    ## HistGradientBoosting ##
    hist_md = HistGradientBoostingClassifier(
        l2_regularization=best_params_hist['l2_regularization'],
        learning_rate=best_params_hist['learning_rate'],
        max_iter=500,
        max_depth=best_params_hist['max_depth'],
        max_bins=best_params_hist['max_bins'],
        min_samples_leaf=best_params_hist['min_samples_leaf'],
        max_leaf_nodes=best_params_hist['max_leaf_nodes'],
        random_state=42
    ).fit(X_train, Y_train)
    
    hist_pred = hist_md.predict_proba(X_test)[:, 1]
    hist_score = roc_auc_score(Y_test, hist_pred)
    print('Fold', i, '==> Hist oof ROC-AUC score is ==>', hist_score)  
    hist_pred_test = hist_md.predict_proba(test_cv)[:, 1]

    ## LGBM ##
    LGBM_md = LGBMClassifier(
        objective='binary',
        n_estimators=500,
        max_depth=best_params_lgbm['max_depth'],
        num_leaves=best_params_lgbm['num_leaves'],
        learning_rate=best_params_lgbm['learning_rate'],
        reg_alpha=best_params_lgbm['reg_alpha'],
        reg_lambda=best_params_lgbm['reg_lambda'],
        subsample=best_params_lgbm['subsample'],
        colsample_bytree=best_params_lgbm['colsample_bytree'],
        random_state=42
    ).fit(X_train, Y_train)

    lgb_pred = LGBM_md.predict_proba(X_test)[:, 1]
    lgb_score = roc_auc_score(Y_test, lgb_pred)
    print('Fold', i, '==> LGBM oof ROC-AUC score is ==>', lgb_score) 
    lgb_pred_test = LGBM_md.predict_proba(test_cv)[:, 1]

    ## XGB ##
    XGB_md = XGBClassifier(
        objective='binary:logistic',
        tree_method='hist',
        n_estimators=500,
        max_depth=best_params_xgb['max_depth'],
        learning_rate=best_params_xgb['learning_rate'],
        gamma=best_params_xgb['gamma'],
        min_child_weight=best_params_xgb['min_child_weight'],
        subsample=best_params_xgb['subsample'],
        colsample_bytree=best_params_xgb['colsample_bytree'],
        reg_alpha=best_params_xgb['reg_alpha'],
        reg_lambda=best_params_xgb['reg_lambda'],
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    ).fit(X_train, Y_train)

    xgb_pred = XGB_md.predict_proba(X_test)[:, 1]
    xgb_score = roc_auc_score(Y_test, xgb_pred)
    print('Fold', i, '==> XGB oof ROC-AUC score is ==>', xgb_score)
    xgb_pred_test = XGB_md.predict_proba(test_cv)[:, 1]

    ## CatBoost ##
    Cat_md = CatBoostClassifier(
        loss_function='Logloss',
        iterations=500,
        depth=best_params_cat['depth'],
        learning_rate=best_params_cat['learning_rate'],
        l2_leaf_reg=best_params_cat['l2_leaf_reg'],
        bagging_temperature=best_params_cat['bagging_temperature'],
        border_count=best_params_cat['border_count'],
        random_strength=best_params_cat['random_strength'],
        verbose=False,
        random_seed=42,
        task_type='CPU'
    ).fit(X_train, Y_train)

    cat_pred = Cat_md.predict_proba(X_test)[:, 1]
    cat_score = roc_auc_score(Y_test, cat_pred)
    print('Fold', i, '==> CatBoost oof ROC-AUC score is ==>', cat_score)
    cat_pred_test = Cat_md.predict_proba(test_cv)[:, 1]    
    
    ## Ensemble ##
    ens_pred_1 = (RF_pred + ET_pred + hist_pred + lgb_pred + xgb_pred + cat_pred) / 6
    ens_pred_2 = (RF_pred_test + ET_pred_test + hist_pred_test + lgb_pred_test + xgb_pred_test + cat_pred_test) / 6
    
    ens_score_fold = roc_auc_score(Y_test, ens_pred_1)
    ens_cv_scores.append(ens_score_fold)
    ens_preds.append(ens_pred_2)
    
    print('Fold', i, '==> Average Ensemble oof ROC-AUC score is ==>', ens_score_fold)
    
    ## Hill Climbing Ensemble ##
    x = pd.DataFrame({'RF': RF_pred,
                      'ET': ET_pred, 
                      'Hist': hist_pred, 
                      'LGBM': lgb_pred,
                      'XGB': xgb_pred,
                      'Cat': cat_pred})
    y = Y_test
        
    x_test = pd.DataFrame({'RF': RF_pred_test,
                           'ET': ET_pred_test, 
                           'Hist': hist_pred_test, 
                           'LGBM': lgb_pred_test,
                           'XGB': xgb_pred_test,
                           'Cat': cat_pred_test})
    
    hill_results = hill_climbing(x, y, x_test)
    
    hill_ens_score_fold = roc_auc_score(y, hill_results[0])
    hill_ens_cv_scores.append(hill_ens_score_fold)
    hill_ens_preds.append(hill_results[1])

    print('Fold', i, '==> Hill Climbing Ensemble oof ROC-AUC score is ==>', hill_ens_score_fold)


In [8]:
print('Average ансамбль ROC-AUC скор на 25 фолдах', np.mean(ens_cv_scores))
print('Hill climbing ансамбль ROC-AUC скор на 25 фолдах', np.mean(hill_ens_cv_scores))

Average ансамбль ROC-AUC скор на 25 фолдах 0.813502417430129
Hill climbing ансамбль ROC-AUC скор на 25 фолдах 0.8491578735779279


In [16]:
RF_pred_train = Cat_md.predict_proba(X)[:, 1]
ET_pred_train = Cat_md.predict_proba(X)[:, 1]
hist_pred_train = Cat_md.predict_proba(X)[:, 1]
lgb_pred_train = Cat_md.predict_proba(X)[:, 1]
xgb_pred_train = Cat_md.predict_proba(X)[:, 1]
cat_pred_train = Cat_md.predict_proba(X)[:, 1]
ens_preds_train = (RF_pred_train + ET_pred_train + hist_pred_train + lgb_pred_train + xgb_pred_train + cat_pred_train) / 6

In [80]:
X_prob = pd.read_csv('../data/train_X.csv').fillna(0) 

def process_diff(X, score, for_train=True, prob_diff_cols=None):
    X_prob = X.copy()
    X_prob['score'] = score

    X_prob['report_date'] = pd.to_datetime(X_prob['report_date'])
    X_prob_sorted = X_prob.sort_values(by=['contract_id', 'report_date'])
    X_prob_sorted = X_prob.drop(['report_date', 'contract_date'], axis=1)
    X_prob_diff = X_prob_sorted.groupby('contract_id').transform(lambda x: x.diff()) 
    X_prob_diff['score_orig'] = score
    X_prob_diff = X_prob_diff.fillna(0)
    if for_train:
        X_prob_diff = X_prob_diff.loc[:, (X_prob_diff != 0).any(axis=0)]
    else:
        X_prob_diff = X_prob_diff[prob_diff_cols]

    # X_prob_with_diff = pd.concat([X_prob_sorted, X_prob_diff.add_suffix('_diff')], axis=1)
    return X_prob_diff, X_prob_diff.columns.tolist()

X_prob_diff, prob_diff_cols = process_diff(X_prob, ens_preds_train, for_train=True)

In [81]:
kfold_roc_aucs = []
kfold_gini = []

sk = StratifiedGroupKFold(n_splits=5, random_state=42, shuffle=True)

for i, (train_idx, test_idx) in enumerate(sk.split(X_prob_diff, Y, groups)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    Y_train, Y_test = Y.iloc[train_idx], Y.iloc[test_idx]
    print('----------------------------------------------------------')
    
    ## CatBoost ##
    cat_diff = CatBoostClassifier(
        loss_function='Logloss',
        iterations=500,
        depth=best_params_cat['depth'],
        learning_rate=best_params_cat['learning_rate'],
        l2_leaf_reg=best_params_cat['l2_leaf_reg'],
        bagging_temperature=best_params_cat['bagging_temperature'],
        border_count=best_params_cat['border_count'],
        random_strength=best_params_cat['random_strength'],
        verbose=False,
        random_seed=42,
        task_type='CPU'
    ).fit(X_train, Y_train)

    cat_pred = cat_diff.predict_proba(X_test)[:, 1]
    cat_score = roc_auc_score(Y_test, cat_pred)
    gini = gini_coefficient(Y_test, cat_pred)
    kfold_gini.append(gini)
    kfold_roc_aucs.append(cat_score)
    
    print('Fold', i, '==> CatBoost oof ROC-AUC score is ==>', cat_score)
    print('Fold', i, '==> CatBoost oof Gini score is ==>', gini)



----------------------------------------------------------
Fold 0 ==> CatBoost oof ROC-AUC score is ==> 0.755980739739129
Fold 0 ==> CatBoost oof Gini score is ==> 0.5119614794782581
----------------------------------------------------------
Fold 1 ==> CatBoost oof ROC-AUC score is ==> 0.8223811537786822
Fold 1 ==> CatBoost oof Gini score is ==> 0.6447623075573643
----------------------------------------------------------
Fold 2 ==> CatBoost oof ROC-AUC score is ==> 0.7515535687864476
Fold 2 ==> CatBoost oof Gini score is ==> 0.5031071375728953
----------------------------------------------------------
Fold 3 ==> CatBoost oof ROC-AUC score is ==> 0.7698035134327064
Fold 3 ==> CatBoost oof Gini score is ==> 0.5396070268654127
----------------------------------------------------------
Fold 4 ==> CatBoost oof ROC-AUC score is ==> 0.7671613485130356
Fold 4 ==> CatBoost oof Gini score is ==> 0.5343226970260713


In [74]:
cat_diff_final = CatBoostClassifier(
        loss_function='Logloss',
        iterations=500,
        depth=best_params_cat['depth'],
        learning_rate=best_params_cat['learning_rate'],
        l2_leaf_reg=best_params_cat['l2_leaf_reg'],
        bagging_temperature=best_params_cat['bagging_temperature'],
        border_count=best_params_cat['border_count'],
        random_strength=best_params_cat['random_strength'],
        verbose=False,
        random_seed=42,
        task_type='CPU'
    ).fit(X_prob_diff, Y)
cat_diff_pred_train = cat_diff_final.predict_proba(X_prob_diff)[:, 1]
print('Fold', i, '==> CatBoost oof ROC-AUC score is ==>', roc_auc_score(Y, cat_diff_pred_train))
print('Fold', i, '==> CatBoost oof Gini score is ==>', gini_coefficient(Y, cat_diff_pred_train))

Fold 4 ==> CatBoost oof ROC-AUC score is ==> 0.9999104177046124
Fold 4 ==> CatBoost oof Gini score is ==> 0.9998208354092248


In [75]:
ens_preds_test_average = pd.DataFrame(ens_preds).apply(np.mean, axis = 0)
ens_preds_test_hill = pd.DataFrame(hill_ens_preds).apply(np.mean, axis = 0)

submit = pd.read_csv('../data/test1_X.csv').fillna(0)[['contract_id', 'report_date']]
submit['score'] = ens_preds_test_hill

In [82]:
test_cv_diff, test_cv_diff_cols = process_diff(pd.read_csv('../data/test1_X.csv').fillna(0), submit['score'], for_train=False, prob_diff_cols=prob_diff_cols)
cat_diff_score = cat_diff.predict_proba(test_cv_diff)[:, 1]


CatBoostError: /Users/zomb-ml-platform-msk/go-agent-21.2.0/pipelines/BuildMaster/catboost.git/catboost/libs/data/model_dataset_compatibility.cpp:81: At position 0 should be feature with name specialization_id (found contract_current_sum).

In [83]:
test_cv_diff

,contract_current_sum,agg_all_contracts__g_contract__bit_da_guid__isMain__count__ALL_TIME,agg_all_contracts__g_contract__abs_change_price_last_ds__isMain__last__ALL_TIME,agg_all_contracts__g_contract__abs_change_price_last_ds__isMain__mean__ALL_TIME,agg_all_contracts__g_contract__rel_change_price_last_ds__isMain__last__ALL_TIME,agg_all_contracts__g_contract__rel_change_price_last_ds__isMain__mean__ALL_TIME,agg_cec_requests__g_contract__request_id__all__count__1W,agg_cec_requests__g_contract__request_id__all__count__2W,agg_cec_requests__g_contract__request_id__all__count__3W,agg_cec_requests__g_contract__request_id__all__count__4W,agg_cec_requests__g_contract__request_id__all__count__5W,agg_cec_requests__g_contract__request_id__all__count__6W,agg_cec_requests__g_contract__request_id__all__count__7W,agg_cec_requests__g_contract__request_id__all__count__8W,agg_cec_requests__g_contract__request_id__all__count__12W,agg_cec_requests__g_contract__request_id__all__count__ALL_TIME,counteragent_sum_agg_cec_requests__g_contract__request_id__all__count__ALL_TIME,agg_cec_requests__g_contract__total_sum_accepted__all__sum__1W,agg_cec_requests__g_contract__total_sum_accepted__all__sum__2W,agg_cec_requests__g_contract__total_sum_accepted__all__sum__3W,agg_cec_requests__g_contract__total_sum_accepted__all__sum__4W,agg_cec_requests__g_contract__total_sum_accepted__all__sum__5W,agg_cec_requests__g_contract__total_sum_accepted__all__sum__6W,agg_cec_requests__g_contract__total_sum_accepted__all__sum__7W,agg_cec_requests__g_contract__total_sum_accepted__all__sum__8W,agg_cec_requests__g_contract__total_sum_accepted__all__sum__12W,agg_cec_requests__g_contract__total_sum_accepted__all__sum__ALL_TIME,counteragent_sum_agg_cec_requests__g_contract__total_sum_accepted__all__sum__ALL_TIME,agg_cec_requests__g_contract__time_btw_requests__all__mean__1M,agg_cec_requests__g_contract__time_btw_requests__all__mean__2M,agg_cec_requests__g_contract__time_btw_requests__all__mean__3M,agg_cec_requests__g_contract__time_btw_requests__all__mean__4M,agg_cec_requests__g_contract__time_btw_requests__all__mean__5M,agg_cec_requests__g_contract__time_btw_requests__all__mean__6M,agg_cec_requests__g_contract__time_btw_requests__all__mean__7M,agg_cec_requests__g_contract__time_btw_requests__all__mean__8M,agg_cec_requests__g_contract__time_btw_requests__all__mean__12M,agg_cec_requests__g_contract__time_btw_requests__all__mean__ALL_TIME,agg_cec_requests__g_contract__created_dt__all__min__ALL_TIME,agg_cec_requests__g_contract__created_dt__accepted__min__ALL_TIME,agg_payments__g_contract__sum__all__countDistinct__1W,agg_payments__g_contract__sum__all__countDistinct__2W,agg_payments__g_contract__sum__all__countDistinct__4W,agg_payments__g_contract__sum__all__countDistinct__8W,agg_payments__g_contract__sum__all__countDistinct__12W,agg_payments__g_contract__sum__all__countDistinct__ALL_TIME,agg_payments__g_contract__sum__all__sum__1W,agg_payments__g_contract__sum__all__sum__2W,agg_payments__g_contract__sum__all__sum__4W,agg_payments__g_contract__sum__all__sum__8W,...,agg_materials__g_contract__order_id__countDistinct__ALL_TIME,agg_materials__g_contract__order_dt__min__ALL_TIME,agg_materials__g_contract__material_type_id__countDistinct__ALL_TIME,agg_materials__g_contract__material_id__countDistinct__ALL_TIME,agg_scontrol__g_contractor__close_delay__defect_type_repair__mean__ALL_TIME,agg_scontrol__g_contractor__close_delay__defect_type_author_supervision__mean__ALL_TIME,agg_scontrol__g_contractor__close_delay__defect_type_GR__mean__ALL_TIME,agg_scontrol__g_contractor__close_delay__defect_type_labour_protection__mean__ALL_TIME,agg_scontrol__g_contractor__close_delay__defect_type_general_contractor__mean__ALL_TIME,agg_scontrol__g_contractor__close_delay__defect_type_arch_supervision__mean__ALL_TIME,agg_scontrol__g_contractor__close_delay__defect_type_tech_supervision__mean__ALL_TIME,agg_scontrol__g_contractor__close_delay__defect_type_app__mean__ALL_TIME,agg_sroomer__g_contractor__sroomer

In [ ]:
submit.to_csv('../submissions/hill_climbin_submission.csv', index=False)